In [1]:
import sys
sys.path.append('/host/d/Github')
import os
import numpy as np
import pandas as pd
import nibabel as nb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import Osteosarcoma.functions_collection as ff
import Osteosarcoma.Build_lists.Build_list as Build_list

import radiomics
from radiomics import (
    featureextractor,  # This module is used for interaction with pyradiomics
)

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

### extract radiomics features based on habitat

In [3]:
# initiate extractor
paramPath = '/host/d/Github/Osteosarcoma/radiomics_settings/MR_setting_habitat_high_ROI.yaml'
extractor = featureextractor.RadiomicsFeatureExtractor(paramPath)

print('Extraction parameters:\n\t', extractor.settings)
print('Enabled filters:\n\t', extractor.enabledImagetypes)
print('Enabled features:\n\t', extractor.enabledFeatures)

Extraction parameters:
	 {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': True, 'normalizeScale': 100, 'removeOutliers': None, 'resampledPixelSpacing': [1, 1, 1], 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True, 'binWidth': 10, 'voxelArrayShift': 300, 'geometryTolerance': 0.0001}
Enabled filters:
	 {'Original': {}, 'LoG': {'sigma': [2.0, 4.0, 6.0]}, 'Wavelet': {}}
Enabled features:
	 {'firstorder': None, 'glcm': ['Autocorrelation', 'JointAverage', 'ClusterProminence', 'ClusterShade', 'ClusterTendency', 'Contrast', 'Correlation', 'DifferenceAverage', 'DifferenceEntropy', 'DifferenceVariance', 'JointEnergy', 'JointEntropy', 'Imc1', 'Imc2', 'Idm', 'Idmn', 'Id', 'Idn', 'InverseVariance', 'MaximumProbability', 'SumEntropy', 'SumSquares'], 'glrlm': None, 'glszm': None, 'gldm': None, 'ngtdm': None}


In [17]:
# define patient list and extract
build = Build_list.Build(os.path.join('/host/d/Data/Habitats/Jishuitan/Patient_lists', 'labels_with_image_info_included.xlsx'))
batch_list, patient_index_list, label_list, image_path_list, mask_path_list = build.__build__()
print(f'Number of cases to process: {len(image_path_list)}')

# rows = []
for i in range(3,len(patient_index_list)):
    cid = patient_index_list[i]
    print('patient_index is', cid)
    img_p = os.path.join('/host/d/Data/Habitats/Jishuitan/resampled_data',str(cid),'img_resampled.nii.gz')
    msk_p = os.path.join('/host/d/projects/Habitats/radiomics/habitats',str(cid),'ROI_high.nii.gz')
    assert os.path.isfile(img_p) == True
    assert os.path.isfile(msk_p) == True
    print('i', i, ' image path:', img_p, 'mask path:', msk_p)

    result = extractor.execute(img_p, msk_p)

    # Keep only radiomics features (drop diagnostics)
    feats = {k: v for k, v in result.items() if not k.startswith("diagnostics_")}
    feats["Patient_index"] = cid
    feats["Image_filepath"] = img_p
    feats["Mask_filepath"] = msk_p

    rows.append(feats)

Number of cases to process: 81
patient_index is 11
i 3  image path: /host/d/Data/Habitats/Jishuitan/resampled_data/11/img_resampled.nii.gz mask path: /host/d/projects/Habitats/radiomics/habitats/11/ROI_high.nii.gz


RuntimeError: Exception thrown in SimpleITK ImageFileReader_Execute: /tmp/SimpleITK-build/ITK/Modules/IO/NIFTI/src/itkNiftiImageIO.cxx:1980:
ITK ERROR: ITK only supports orthonormal direction cosines.  No orthonormal definition found!

In [10]:
df = pd.DataFrame(rows)

# Put id/path columns first
front_cols = [c for c in ["Patient_index", "Image_filepath", "Mask_filepath"] if c in df.columns]
other_cols = [c for c in df.columns if c not in front_cols]
df = df[front_cols + other_cols]

df.to_excel('/host/d/projects/Habitats/radiomics/habitats/radiomics_measurements_habitats.xlsx', index=False)